# Deep Past Initiative — ByT5 (Participant 1: Byte-Level Seq2Seq)


## 1. Install Dependencies


In [ ]:
# Match mbart50.ipynb: offline mode is the default and makes no network requests.
EXECUTION_MODE = "offline"  # change to "online" only when internet access is enabled
assert EXECUTION_MODE in ("online", "offline")
IS_ONLINE = EXECUTION_MODE == "online"
print(f"Running in {EXECUTION_MODE.upper()} mode")

import sys
import subprocess

if IS_ONLINE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers", "datasets", "accelerate", "sentencepiece",
         "mlflow", "huggingface_hub"],
        check=True,
    )
else:
    print("Offline mode: skipping pip install. Use the Kaggle image's preinstalled packages.")


## 2. Imports


In [ ]:
import os
import sys
import glob
import gc
import json
import random
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)



## 3. Configuration and Offline Mode

`EXECUTION_MODE = "offline"` is the default. It skips package installation, tracking, Hugging Face login, and model downloads. Before running offline, add the saved output of [nikhilsatani/byt5-small](https://www.kaggle.com/code/nikhilsatani/byt5-small/notebook) through Kaggle **Add Input → Notebooks**. The selected notebook version must expose the pretrained ByT5 files in its output. The notebook searches `/kaggle/input` for a matching T5 checkpoint. If multiple checkpoints are attached, set `BASE_MODEL_OFFLINE_PATH` to the exact folder containing `config.json` and the model weights.

Use `EXECUTION_MODE = "online"` only with Kaggle Internet enabled; that mode can download the base checkpoint. Never enable `USE_TRACKING` or `PUBLISH_MODELS` during an offline run.


## 3.1. Experiment Tracking with MLflow + DagsHub


In [ ]:
# ============================================================
# EXPERIMENT TRACKING
#
# True:
#   - Read Kaggle Secrets (HF_TOKEN, MLflow credentials)
#   - Log runs to MLflow / DagsHub
#   - Push each trained model + tokenizer to a private Hugging Face Hub repo
#
# False:
#   - Do NOT read any Secrets
#   - Do NOT use MLflow / DagsHub
#   - Do NOT push anything to Hugging Face
#   - Save everything locally only
#
# For reproducibility use False and turn on any GPU environment
# ============================================================
USE_TRACKING = False
PUBLISH_MODELS = False     # True: upload trained weights to Kaggle Models (needs Internet ON)
KAGGLE_USERNAME = "eeee13"  # owner of the Kaggle Models read by ensemble-pipeline.ipynb

# ============================================================
# PLACEHOLDERS — replace with real values before running
# ============================================================
AUTHOR_NAME = "name"
HF_USERNAME = "userName"

HF_TOKEN = None
MLFLOW_TRACKING_URI = None
MLFLOW_EXPERIMENT_NAME = "Byte-Level-Seq2Seq-machine-translation"

if USE_TRACKING and not IS_ONLINE:
    raise ValueError("USE_TRACKING requires EXECUTION_MODE = 'online'.")
if PUBLISH_MODELS and not IS_ONLINE:
    raise ValueError("PUBLISH_MODELS requires EXECUTION_MODE = 'online'.")

if USE_TRACKING:
    import mlflow
    from huggingface_hub import login as hf_login
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

    os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("MLFLOW_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("MLFLOW_PASSWORD")

    MLFLOW_TRACKING_URI = user_secrets.get_secret("MLFLOW_TRACKING_URI")

# Load competition data directly; no pipeline or normalization here
TRAIN_DATA_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv"
TEST_DATA_PATH  = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
OUTPUT_DIR = "outputs"
MODEL_ARCHITECTURE_TAG = "byt5"
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42  # seed study: run the notebook with SEED = 42, 43 and 44
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if USE_TRACKING:
    if HF_TOKEN:
        hf_login(token=HF_TOKEN)

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Tracking: {USE_TRACKING}")

## Shared split, metric and run record (Phase 2)

The next four cells are identical in all four notebooks (copied from `common/` by `tools/sync_shared_cells.py`). They give every model **the same 156 validation documents** (checked by a SHA-256 of the sorted ids), **the same metric** (sacreBLEU-identical BLEU and chrF++ with word n-grams, `tests/test_metrics.py`), write one prediction file plus one JSON run record per model into `results/`, and record the package versions and GPU (`results/env_*.json`).

In [ ]:
# >>> SHARED CELL: common/split.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import hashlib
import random

ID_COL = "oare_id"
VAL_FRACTION = 0.1
SPLIT_SEED = 42
# sha256 of "\n".join(sorted(val_ids)) for the competition train.csv (1561 documents -> 156 val).
EXPECTED_VAL_SHA256 = "1939161cfcb9ff85082f20b36be2362da429f1da14f8a94f289249ffb78e1304"
EXPECTED_N_VAL = 156


def val_ids_sha256(val_ids):
    return hashlib.sha256("\n".join(sorted(map(str, val_ids))).encode("utf-8")).hexdigest()


def make_split(df, id_col=ID_COL, val_fraction=VAL_FRACTION, seed=SPLIT_SEED, check=True):
    """Return (train_df, val_df), split by document id. Rows with empty text are dropped first."""
    df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
    doc_ids = sorted(df[id_col].astype(str).unique().tolist())
    random.Random(seed).shuffle(doc_ids)
    n_val = max(1, int(len(doc_ids) * val_fraction))
    val_ids = set(doc_ids[:n_val])
    is_val = df[id_col].astype(str).isin(val_ids)
    train_df = df[~is_val].reset_index(drop=True)
    val_df = df[is_val].reset_index(drop=True)
    digest = val_ids_sha256(val_ids)
    print(f"[shared split] train={len(train_df)} rows, val={len(val_df)} rows "
          f"({n_val} documents), val_ids_sha256={digest[:16]}...")
    if check:
        assert n_val == EXPECTED_N_VAL and digest == EXPECTED_VAL_SHA256, (
            "Validation split differs from the shared reference split "
            f"(n_val={n_val}, sha256={digest}). Is train.csv the original competition file?")
    assert not set(train_df[id_col].astype(str)) & val_ids, "document leakage between train and val"
    return train_df, val_df

In [ ]:
# >>> SHARED CELL: common/metrics.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import math
import re
from collections import Counter

# ---------------------------------------------------------------- BLEU (13a)
_TOK_13A_RULES = [
    (re.compile(r'([\{-\~\[-\` -\&\(-\+\:-\@\/])'), r' \1 '),
    (re.compile(r'([^0-9])([\.,])'), r'\1 \2 '),
    (re.compile(r'([\.,])([^0-9])'), r' \1 \2'),
    (re.compile(r'([0-9])(-)'), r'\1 \2 '),
]


def _tokenize_13a(line):
    line = line.replace('<skipped>', '').replace('-\n', '').replace('\n', ' ')
    if '&' in line:
        line = (line.replace('&quot;', '"').replace('&amp;', '&')
                    .replace('&lt;', '<').replace('&gt;', '>'))
    line = f' {line} '
    for pattern, repl in _TOK_13A_RULES:
        line = pattern.sub(repl, line)
    return ' '.join(line.split())


def _word_ngrams(tokens, max_order):
    return Counter(tuple(tokens[i:i + n])
                   for n in range(1, max_order + 1)
                   for i in range(len(tokens) - n + 1))


def bleu_stats(hyp, ref, max_order=4):
    """Sufficient statistics of one segment: [hyp_len, ref_len, correct_1..4, total_1..4]."""
    h = _tokenize_13a(hyp.rstrip()).split()
    r = _tokenize_13a(ref.rstrip()).split()
    h_ng, r_ng = _word_ngrams(h, max_order), _word_ngrams(r, max_order)
    correct, total = [0] * max_order, [0] * max_order
    for ng, c in h_ng.items():
        total[len(ng) - 1] += c
        if ng in r_ng:
            correct[len(ng) - 1] += min(c, r_ng[ng])
    return [len(h), len(r)] + correct + total


def bleu_from_stats(stats, max_order=4):
    sys_len, ref_len = stats[0], stats[1]
    correct, total = stats[2:2 + max_order], stats[2 + max_order:]
    bp = 1.0
    if sys_len < ref_len:
        bp = math.exp(1 - ref_len / sys_len) if sys_len > 0 else 0.0
    if not any(correct):
        return 0.0
    precisions = [0.0] * max_order
    smooth = 1.0
    for n in range(max_order):
        if total[n] == 0:
            break
        if correct[n] == 0:
            smooth *= 2
            precisions[n] = 100.0 / (smooth * total[n])
        else:
            precisions[n] = 100.0 * correct[n] / total[n]
    log_sum = sum(-9999999999 if p == 0 else math.log(p) for p in precisions)
    return bp * math.exp(log_sum / max_order)


# ---------------------------------------------------------------- chrF++
_PUNCTS = set('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')


def _chrf_words(sent):
    out = []
    for w in sent.split():
        if len(w) == 1:
            out.append(w)
        elif w[-1] in _PUNCTS:
            out += [w[:-1], w[-1]]
        elif w[0] in _PUNCTS:
            out += [w[0], w[1:]]
        else:
            out.append(w)
    return out


def _chrf_ngrams(sent, char_order, word_order):
    chars = ''.join(sent.split())
    grams = [Counter(chars[i:i + n] for i in range(len(chars) - n + 1))
             for n in range(1, char_order + 1)]
    words = _chrf_words(sent)
    grams += [Counter(' '.join(words[i:i + n]) for i in range(len(words) - n + 1))
              for n in range(1, word_order + 1)]
    return grams


def chrf_stats(hyp, ref, char_order=6, word_order=2):
    """Flattened [hyp, ref, match] counts for every char then word n-gram order."""
    stats = []
    for h, r in zip(_chrf_ngrams(hyp, char_order, word_order),
                    _chrf_ngrams(ref, char_order, word_order)):
        match = sum(min(c, r[g]) for g, c in h.items() if g in r)
        stats += [sum(h.values()) if r else 0, sum(r.values()), match]
    return stats


def chrf_from_stats(stats, beta=2):
    factor = beta ** 2
    avg_p = avg_r = 0.0
    eff = 0
    for i in range(len(stats) // 3):
        n_hyp, n_ref, n_match = stats[3 * i:3 * i + 3]
        if n_hyp > 0 and n_ref > 0:
            avg_p += n_match / n_hyp
            avg_r += n_match / n_ref
            eff += 1
    if eff == 0:
        return 0.0
    avg_p, avg_r = avg_p / eff, avg_r / eff
    if avg_p + avg_r == 0:
        return 0.0
    return 100.0 * (1 + factor) * avg_p * avg_r / (factor * avg_p + avg_r)


# ---------------------------------------------------------------- public API
def _sum_stats(rows):
    return [sum(col) for col in zip(*rows)] if rows else None


def segment_stats(preds, refs):
    """Per-segment (bleu_stats, chrf_stats); lets bootstrap resample without recomputing n-grams."""
    preds = ['' if p is None else str(p) for p in preds]
    refs = ['' if r is None else str(r) for r in refs]
    assert len(preds) == len(refs), f"{len(preds)} predictions vs {len(refs)} references"
    return [bleu_stats(p, r) for p, r in zip(preds, refs)], [chrf_stats(p, r) for p, r in zip(preds, refs)]


def scores_from_segment_stats(bleu_rows, chrf_rows):
    bleu = bleu_from_stats(_sum_stats(bleu_rows))
    chrfpp = chrf_from_stats(_sum_stats(chrf_rows))
    return {"bleu": bleu, "chrfpp": chrfpp, "geo_mean": math.sqrt(max(bleu, 0.0) * max(chrfpp, 0.0))}


def corpus_scores(preds, refs):
    """Corpus BLEU, chrF++ and their geometric mean (the competition score), all on 0-100."""
    return scores_from_segment_stats(*segment_stats(preds, refs))


def sentence_chrfpp(hyp, ref):
    """Sentence-level chrF++, used as the MBR utility."""
    return chrf_from_stats(chrf_stats(hyp, ref))


METRIC_SIGNATURE = "BLEU=sacrebleu-2.x-13a-exp | chrF++=sacrebleu-2.x-c6-w2-b2 | geo_mean=sqrt(BLEU*chrF++)"

In [ ]:
# >>> SHARED CELL: common/runlog.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import datetime
import json
import os

import pandas as pd


RESULTS_DIR = "results"


def finalize_run(system, seed, ids, refs, preds, hyperparameters, log_history=None,
                 out_dir=RESULTS_DIR, notes=None):
    os.makedirs(out_dir, exist_ok=True)
    ids = [str(i) for i in ids]
    refs = ["" if r is None else str(r) for r in refs]
    preds = ["" if p is None else str(p) for p in preds]
    scores = corpus_scores(preds, refs)
    tag = f"{system}__seed{seed}"
    pred_path = os.path.join(out_dir, f"preds_val_{tag}.csv")
    pd.DataFrame({"id": ids, "target_text": refs, "pred_translation": preds}).to_csv(pred_path, index=False)
    record = {
        "system": system,
        "seed": seed,
        "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
        "metric": METRIC_SIGNATURE,
        "n_val_docs": len(ids),
        "val_ids_sha256": val_ids_sha256(ids),
        "final_val": scores,
        "hyperparameters": hyperparameters,
        "notes": notes,
        "log_history": log_history or [],
    }
    with open(os.path.join(out_dir, f"run_{tag}.json"), "w") as f:
        json.dump(record, f, indent=2, default=str, ensure_ascii=False)
    print(f"[run record] {tag}: BLEU={scores['bleu']:.2f} chrF++={scores['chrfpp']:.2f} "
          f"GM={scores['geo_mean']:.2f} -> {pred_path}")
    return scores

In [ ]:
# >>> SHARED CELL: common/env.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import importlib
import json
import os
import platform
import sys

TRACKED_PACKAGES = ["torch", "transformers", "datasets", "accelerate", "peft", "bitsandbytes",
                    "sentencepiece", "sacrebleu", "pandas", "numpy", "scikit-learn"]


def report_environment(notebook, out_dir="results"):
    versions = {}
    for name in TRACKED_PACKAGES:
        try:
            module = importlib.import_module("sklearn" if name == "scikit-learn" else name)
            versions[name] = getattr(module, "__version__", "unknown")
        except Exception:
            versions[name] = None
    env = {"notebook": notebook, "python": sys.version.split()[0], "platform": platform.platform(),
           "packages": versions, "gpu": None, "cuda": None}
    try:
        import torch
        if torch.cuda.is_available():
            env["gpu"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
            env["cuda"] = torch.version.cuda
    except Exception:
        pass
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, f"env_{notebook}.json"), "w") as f:
        json.dump(env, f, indent=2)
    print(json.dumps(env, indent=2))
    return env

In [ ]:
report_environment("byt5")  # -> results/env_byt5.json

## 4. Load Dataset


In [ ]:
df = pd.read_csv(TRAIN_DATA_PATH)

# Shared document-level split (identical in every notebook, verified by checksum).
train_df, val_df = make_split(df)
test_df = pd.read_csv(TEST_DATA_PATH)
print(f"test: {len(test_df)} rows")

## 5. Train/Validation Split and Ensemble Subsets

The validation split is fixed by the shared pipeline and should remain identical across team members.

Because the dataset is currently small, each ensemble model uses a different 85% random subset of the training data. This provides model diversity without reducing each training set too aggressively.

When the dataset becomes substantially larger, consider switching to non-overlapping k-fold splits.


In [ ]:
# Model 1 is the ByT5 baseline and is trained on the FULL training split (same data as mBART/Qwen).
# Models 2..N are extra members for the ByT5-only MBR ensemble, each on a random 85% subset.
# Seed study (SEED 43/44): only the baseline is trained.
N_MODELS = 2 if SEED == 42 else 1
BOOTSTRAP_FRACTION = 0.85  # used only by members 2..N



def make_bootstrap_splits(train_df, n_models, fraction, seed):
    """Train each model on a different random subset of the training data."""
    splits = []
    rng = np.random.default_rng(seed)
    n = len(train_df)
    for i in range(n_models):
        if i == 0:
            splits.append(train_df.copy())  # baseline: full training split
            continue
        idx = rng.choice(n, size=int(n * fraction), replace=False)
        splits.append(train_df.iloc[idx].reset_index(drop=True))
    return splits


train_splits = make_bootstrap_splits(train_df, N_MODELS, BOOTSTRAP_FRACTION, SEED)
for i, s in enumerate(train_splits):
    print(f"Модель {i + 1}: train size = {len(s)}")

## 6. Tokenizer and Preprocessing

ByT5 operates at the byte level, so it handles the special characters used in cuneiform transliteration without a custom vocabulary.

For the current dataset size, `google/byt5-small` is used to reduce overfitting and GPU memory usage. Consider `byt5-base` after the dataset grows.


In [ ]:
MODEL_NAME = "google/byt5-small"   # canonical model id
BASE_MODEL_OFFLINE_PATH = ""  # optional: set the ByT5 folder from the Notebook input output
# The notebook scans /kaggle/input and detects one ByT5 checkpoint automatically.
# If several T5/ByT5 inputs are attached, use the full path to the intended folder here.

def _is_byt5_checkpoint(model_dir):
    if not os.path.isfile(os.path.join(model_dir, "config.json")):
        return False
    try:
        with open(os.path.join(model_dir, "config.json"), encoding="utf-8") as f:
            config = json.load(f)
    except (OSError, json.JSONDecodeError):
        return False
    if config.get("model_type") != "t5" or int(config.get("vocab_size", 0)) != 384:
        return False
    has_weights = any(os.path.isfile(os.path.join(model_dir, name)) for name in (
        "model.safetensors", "pytorch_model.bin", "model.safetensors.index.json",
        "pytorch_model.bin.index.json",
    ))
    has_shards = bool(glob.glob(os.path.join(model_dir, "model-*.safetensors")) or
                      glob.glob(os.path.join(model_dir, "pytorch_model-*.bin")))
    return has_weights or has_shards


def resolve_model_source():
    if IS_ONLINE:
        return MODEL_NAME

    if BASE_MODEL_OFFLINE_PATH and _is_byt5_checkpoint(BASE_MODEL_OFFLINE_PATH):
        print(f"Using attached ByT5 checkpoint: {BASE_MODEL_OFFLINE_PATH}")
        return BASE_MODEL_OFFLINE_PATH

    input_root = "/kaggle/input"
    candidates = set()
    if os.path.isdir(input_root):
        for config_path in glob.glob(os.path.join(input_root, "**", "config.json"), recursive=True):
            model_dir = os.path.dirname(config_path)
            path_lower = model_dir.lower()
            if "akkadian-byt5" in path_lower or "final_model" in path_lower or "checkpoint-" in path_lower:
                continue
            if _is_byt5_checkpoint(model_dir):
                candidates.add(os.path.realpath(model_dir))

    if len(candidates) == 1:
        found = candidates.pop()
        print(f"Auto-detected attached ByT5 checkpoint: {found}")
        return found
    if len(candidates) > 1:
        options = "\n".join(sorted(candidates))
        raise RuntimeError(
            "More than one ByT5 checkpoint is attached. Set BASE_MODEL_OFFLINE_PATH to the "
            "pretrained google/byt5-small directory:\n" + options
        )

    # A pre-populated local Hugging Face cache is also safe: from_pretrained below is
    # explicitly called with local_files_only=True in offline mode.
    return MODEL_NAME


MODEL_SOURCE = resolve_model_source()
MAX_SOURCE_LEN = 1024  # UTF-8 bytes: p95 of transliterations = 1024
MAX_TARGET_LEN = 1408  # UTF-8 bytes: p95 of translations = 1384

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, local_files_only=not IS_ONLINE)
except (OSError, ValueError) as exc:
    if not IS_ONLINE:
        raise RuntimeError(
            "ByT5 base weights/tokenizer are not available locally. Add the saved output of "
            "nikhilsatani/byt5-small through Kaggle Add Input -> Notebooks and make sure "
            "that notebook version exposes the base checkpoint files. If needed, set "
            "BASE_MODEL_OFFLINE_PATH to its mounted directory. No internet download was attempted."
        ) from exc
    raise

def preprocess(examples):
    model_inputs = tokenizer(
        examples["transliteration"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["translation"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

## 7. Metrics (BLEU, chrF++)


## 8. Fine-Tuning

Each model is trained and evaluated, then saved locally. Offline runs require no Hub credentials and make no network requests. Optional Hugging Face tracking/upload is available only when `USE_TRACKING = True` and `EXECUTION_MODE = "online"`.


In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    s = corpus_scores(decoded_preds, decoded_labels)  # shared metric (sacreBLEU BLEU + chrF++)
    return {"bleu": s["bleu"], "chrfpp": s["chrfpp"], "geom_mean": s["geo_mean"]}

In [ ]:
def train_one_model(model_idx, train_subset_df, val_df, config):
    run_name = f"byt5_model_{model_idx + 1}"

    run_context = mlflow.start_run(run_name=run_name) if USE_TRACKING else nullcontext(None)

    with run_context:
        if USE_TRACKING:
            mlflow.log_params({
                "learning_rate": config["learning_rate"],
                "batch_size": config["batch_size"],
                "max_length": MAX_SOURCE_LEN,
                "epochs": config["epochs"],
                "lora_r": None,       # Not used for ByT5 full fine-tuning; kept for consistency with the LoRA setup
                "lora_alpha": None,
                "model_idx": model_idx + 1,
                "base_model": MODEL_NAME,
            })
            mlflow.set_tag("model_architecture", MODEL_ARCHITECTURE_TAG)
            mlflow.set_tag("author", AUTHOR_NAME)  # Code runner

        try:
            model = AutoModelForSeq2SeqLM.from_pretrained(
                MODEL_SOURCE, local_files_only=not IS_ONLINE
            )
        except (OSError, ValueError) as exc:
            if not IS_ONLINE:
                raise RuntimeError(
                    "ByT5 model weights are unavailable locally. Attach the pretrained "
                    "google/byt5-small checkpoint and set BASE_MODEL_OFFLINE_PATH correctly."
                ) from exc
            raise

        train_ds = Dataset.from_pandas(train_subset_df[["transliteration", "translation"]])
        eval_ds = Dataset.from_pandas(val_df[["transliteration", "translation"]])

        train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
        eval_ds = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

        data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

        training_args = Seq2SeqTrainingArguments(
            output_dir=f"{OUTPUT_DIR}/model_{model_idx + 1}",
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=config["batch_size"],
            gradient_accumulation_steps=config.get("grad_accum", 1),
            gradient_checkpointing=True,  # long byte sequences; trades speed for memory
            learning_rate=config["learning_rate"],
            num_train_epochs=config["epochs"],
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            predict_with_generate=True,
            generation_max_length=MAX_TARGET_LEN,
            generation_num_beams=config.get("num_beams_eval", 4),
            load_best_model_at_end=True,
            metric_for_best_model="geom_mean",
            greater_is_better=True,
            logging_steps=10,
            seed=SEED + model_idx,
            report_to=[],  # Logged to MLflow manually below
            fp16=torch.cuda.is_available(),
        )

        # Newer Transformers versions use `processing_class` instead of `tokenizer`
        import inspect
        trainer_kwargs = dict(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )
        if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters:
            trainer_kwargs["processing_class"] = tokenizer
        else:
            trainer_kwargs["tokenizer"] = tokenizer

        trainer = Seq2SeqTrainer(**trainer_kwargs)

        train_result = trainer.train()
        eval_metrics = trainer.evaluate()

        if USE_TRACKING:
            mlflow.log_metrics({
                "train_loss": train_result.training_loss,
                "val_loss": eval_metrics.get("eval_loss", 0.0),
                "val_bleu": eval_metrics.get("eval_bleu", 0.0),
                "val_chrfpp": eval_metrics.get("eval_chrfpp", 0.0),
                "val_geom_mean": eval_metrics.get("eval_geom_mean", 0.0),
            })

        # --- Save weights locally for ensembling without downloading again ---
        local_path = f"{OUTPUT_DIR}/final_model_{model_idx + 1}"
        trainer.save_model(local_path)
        tokenizer.save_pretrained(local_path)
        with open(f"{local_path}/log_history.json", "w") as f:  # per-epoch metrics for the run record
            json.dump(trainer.state.log_history, f, indent=2, default=str)

        # --- Upload weights to a private Hugging Face Hub repository ---
        repo_name = f"{HF_USERNAME}/akkadian-byt5-model{model_idx + 1}"
        if USE_TRACKING:
            model.push_to_hub(repo_name, private=True)
            tokenizer.push_to_hub(repo_name, private=True)
            mlflow.set_tag("hf_repo", f"https://huggingface.co/{repo_name}")
            mlflow.set_tag("local_path", local_path)

        chrf_score = eval_metrics.get("eval_chrfpp", 0)
        bleu_score = eval_metrics.get("eval_bleu", 0)
        geom_mean_score = eval_metrics.get("eval_geom_mean", 0)
        print(f"[Model {model_idx + 1}] val_chrfpp={chrf_score:.2f}  val_BLEU={bleu_score:.2f}  val_GM={geom_mean_score:.2f}  "
              f"-> {repo_name if USE_TRACKING else local_path} (local: {local_path})")

        # --- Free GPU memory before training the next model ---
        del trainer
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return repo_name, local_path

## 9. Train 2–3 Models

Different hyperparameters and training subsets provide the diversity needed for the ensemble.

Adjust `MODEL_CONFIGS` to match your available compute budget.

Models are trained sequentially to avoid accumulating multiple models in GPU memory.


In [ ]:
MODEL_CONFIGS = [
    # "name" is the system name used in results/ (preds_val_<name>__seed<SEED>.csv)
    # Effective batch 8 (2 x 4). Checkpoint selection uses greedy decoding (num_beams_eval=1), like mBART;
    # early stopping (patience 3) ends training once the validation geometric mean stops improving.
    {"name": "byt5", "learning_rate": 2e-4, "batch_size": 2, "grad_accum": 4, "epochs": 20, "num_beams_eval": 1},
    {"name": "byt5-member2", "learning_rate": 2e-4, "batch_size": 2, "grad_accum": 4, "epochs": 15, "num_beams_eval": 1},
][:N_MODELS]

trained_model_paths = []  # [(repo_name, local_path), ...]
for i in range(N_MODELS):
    repo_name, local_path = train_one_model(i, train_splits[i], val_df, MODEL_CONFIGS[i])
    trained_model_paths.append((repo_name, local_path))

## 10. Inference with Beam Search and Top-K Hypotheses

For each input, generate multiple beam-search candidates for `pred_top_k` and MBR ensembling.

Models are loaded one at a time, preferring the local path and falling back to Hugging Face Hub.


In [ ]:
NUM_BEAMS = 8
NUM_RETURN_SEQUENCES = 5  # top-K


def load_model_for_inference(repo_name, local_path):
    """Load one model from the local path or Hugging Face Hub."""
    source = local_path if os.path.isdir(local_path) else repo_name
    model = AutoModelForSeq2SeqLM.from_pretrained(source)
    model.to(device)
    model.eval()
    return model


def unload_model(model):
    """Unload the model before loading the next one."""
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def generate_predictions(model, dataset_df, num_beams=NUM_BEAMS, num_return_sequences=NUM_RETURN_SEQUENCES,
                          batch_size=4):
    texts = dataset_df["transliteration"].tolist()
    all_top1, all_topk = [], []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts, return_tensors="pt", truncation=True,
            max_length=MAX_SOURCE_LEN, padding=True,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LEN,
                num_beams=num_beams,
                num_return_sequences=num_return_sequences,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        # Outputs are grouped by num_return_sequences for each input
        for i in range(len(batch_texts)):
            cand = decoded[i * num_return_sequences:(i + 1) * num_return_sequences]
            all_topk.append(cand)
            all_top1.append(cand[0])

    return all_top1, all_topk

## 11. Unified Prediction File Format

Each model produces validation and test prediction files. `pred_top_k` is stored as a JSON list for compatibility with the team pipeline.


In [ ]:
def save_predictions(df_subset, top1_preds, topk_preds, model_name, split, id_col="sentence_id", has_target=True):
    data = {
        "id": df_subset[id_col].tolist(),
        "source_text": df_subset["transliteration"].tolist(),
        "pred_translation": top1_preds,
        "pred_top_k": [json.dumps(k, ensure_ascii=False) for k in topk_preds],
    }
    if has_target and "translation" in df_subset.columns:
        data["target_text"] = df_subset["translation"].tolist()

    out_df = pd.DataFrame(data)
    fname = f"{OUTPUT_DIR}/preds_{split}_{model_name}.csv"
    out_df.to_csv(fname, index=False)
    print(f"Сохранено: {fname}")
    return fname

In [ ]:
# One name per trained model. (Before Phase 2 this was range(N_MODELS - 1), so only the first
# model was used for predictions and the MBR ensemble.)
model_names = [cfg["name"] for cfg in MODEL_CONFIGS]

val_preds_per_model, test_preds_per_model = {}, {}
for (repo_name, local_path), mname, cfg in zip(trained_model_paths, model_names, MODEL_CONFIGS):
    model = load_model_for_inference(repo_name, local_path)

    top1, topk = generate_predictions(model, val_df)
    val_preds_per_model[mname] = {"top1": top1, "topk": topk}
    save_predictions(val_df, top1, topk, mname, "val", id_col="oare_id", has_target=True)
    log_path = f"{local_path}/log_history.json"
    finalize_run(
        system=mname, seed=SEED, ids=val_df["oare_id"], refs=val_df["translation"], preds=top1,
        hyperparameters={**cfg, "base_model": MODEL_NAME, "model_source": MODEL_SOURCE, "max_source_len": MAX_SOURCE_LEN,
                         "max_target_len": MAX_TARGET_LEN, "bootstrap_fraction": BOOTSTRAP_FRACTION,
                         "train_rows": len(train_splits[model_names.index(mname)]),
                         "inference_num_beams": NUM_BEAMS, "trainer_seed": SEED + model_names.index(mname)},
        log_history=json.load(open(log_path)) if os.path.exists(log_path) else None,
    )

    test_top1, test_topk = generate_predictions(model, test_df)
    test_preds_per_model[mname] = {"top1": test_top1, "topk": test_topk}
    save_predictions(test_df, test_top1, test_topk, mname, "test", id_col="id", has_target=False)
    unload_model(model)

## 13. MBR Ensemble (Minimum Bayes Risk)

For each example, combine the top-K hypotheses from all models and select the candidate with the highest average pairwise utility.


In [ ]:
def chrf_utility(hyp, ref):
    return sentence_chrfpp(hyp, ref)  # shared sentence-level chrF++


def mbr_select(candidates, utility_fn):
    """Return the candidate with the highest average pairwise similarity."""
    best_hyp, best_score = candidates[0], -1.0
    for hyp in candidates:
        others = [c for c in candidates if c != hyp]
        avg_score = np.mean([utility_fn(hyp, o) for o in others]) if others else 0.0
        if avg_score > best_score:
            best_score, best_hyp = avg_score, hyp
    return best_hyp, best_score


def ensemble_mbr(preds_per_model, n_examples):
    ensembled = []
    for idx in range(n_examples):
        pooled = []
        for mname in preds_per_model:
            pooled.extend(preds_per_model[mname]["topk"][idx])
        best_hyp, _ = mbr_select(pooled, chrf_utility)
        ensembled.append(best_hyp)
    return ensembled

In [ ]:
if len(model_names) < 2:
    print("Single model (seed study run) — no ByT5 MBR ensemble.")
else:
    # --- Validation ensemble and metrics ---
    ensembled_val = ensemble_mbr(val_preds_per_model, len(val_df))

    ens_scores = finalize_run(
        system="byt5-mbr", seed=SEED, ids=val_df["oare_id"], refs=val_df["translation"], preds=ensembled_val,
        hyperparameters={"members": model_names, "num_beams": NUM_BEAMS,
                         "num_return_sequences": NUM_RETURN_SEQUENCES, "utility": "sentence chrF++"},
    )

    # Compare with each individual model (same shared metric, full references)
    for mname in model_names:
        s = corpus_scores(val_preds_per_model[mname]["top1"], val_df["translation"].tolist())
        print(f"{mname} (val, single-best beam) — BLEU: {s['bleu']:.2f}, chrF++: {s['chrfpp']:.2f}, GM: {s['geo_mean']:.2f}")

In [ ]:
# --- Test predictions (submission) ---
if len(model_names) >= 2:
    final_test = ensemble_mbr(test_preds_per_model, len(test_df))   # ByT5 MBR ensemble
else:
    final_test = test_preds_per_model[model_names[0]]["top1"]       # single model

pd.DataFrame({
    "id": test_df["id"],
    "source_text": test_df["transliteration"],
    "pred_translation": final_test,
}).to_csv(f"{OUTPUT_DIR}/preds_test_ensemble.csv", index=False)
pd.DataFrame({"id": test_df["id"], "translation": final_test}).to_csv("submission.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/preds_test_ensemble.csv and submission.csv")

## 14. Log Ensemble Results to MLflow


In [ ]:
if USE_TRACKING and len(model_names) >= 2:
    with mlflow.start_run(run_name="byt5_ensemble_mbr2"):
        mlflow.log_params({
            "n_models": N_MODELS,
            "num_beams": NUM_BEAMS,
            "num_return_sequences": NUM_RETURN_SEQUENCES,
            "ensemble_method": "MBR (chrfpp utility)",
        })
        mlflow.log_metrics({
            "val_bleu": ens_scores["bleu"],
            "val_chrfpp": ens_scores["chrfpp"],
            "val_geom_mean": ens_scores["geo_mean"],
        })
        mlflow.set_tag("model_architecture", "byt5-ensemble-mbr")
        mlflow.set_tag("author", AUTHOR_NAME)  # Code runner
        for (repo_name, local_path), mname in zip(trained_model_paths, model_names):
            mlflow.set_tag(f"hf_repo_{mname}", f"https://huggingface.co/{repo_name}")
            mlflow.set_tag(f"local_path_{mname}", local_path)
else:
    print("Tracking disabled — skipping MLflow ensemble logging.")

## 15. Save model to Kaggle

In [ ]:
# Optional upload to Kaggle Models. Offline training leaves this disabled and needs no Kaggle API call.
# To publish, enable Internet, set EXECUTION_MODE="online" and PUBLISH_MODELS=True in the setup/config cells,
# then run this cell while the saved outputs/final_model_* directories are available.
if PUBLISH_MODELS:
    import kagglehub

    models_to_publish = [("byt5", f"{OUTPUT_DIR}/final_model_1")]
    member2_path = f"{OUTPUT_DIR}/final_model_2"
    if SEED == 42 and os.path.isdir(member2_path):
        models_to_publish.append(("byt5-member2", member2_path))

    for mname, local_path in models_to_publish:
        if not os.path.isdir(local_path):
            raise FileNotFoundError(f"Cannot publish; model directory is missing: {local_path}")
        handle = f"{KAGGLE_USERNAME}/akkadian-byt5/pytorch/{mname}-seed{SEED}"
        kagglehub.model_upload(
            handle=handle,
            local_model_dir=local_path,
            version_notes=f"{mname}, seed {SEED}, shared split (Phase 2)",
            license_name="Apache 2.0",
        )
        print("Uploaded:", handle)
else:
    print(f"PUBLISH_MODELS = False — weights stay in {OUTPUT_DIR}/final_model_*")
